# Web Scraping IMDB Reviews - Tutorial Lengkap

Notebook ini mengajarkan web scraping dari dasar, pakai review film IMDB.

## Apa itu Web Scraping?

Web scraping = teknik mengambil data dari website secara otomatis menggunakan program.

Bayangkan kamu mau copy 500 review dari IMDB. Kalau manual, buka satu-satu lalu copy-paste. Capek kan? Dengan scraping, Python yang kerjain.

## Alur Scraping

```
1. Buka halaman web (Request)
2. Baca kode HTML-nya (Parse)
3. Cari data yang mau diambil (Extract)
4. Simpan datanya (Store)
```

## Tools

- **requests** -> buka halaman web, ambil HTML-nya
- **BeautifulSoup** -> baca HTML, cari elemen yang kita mau
- **pandas** -> simpan data ke tabel/CSV

---
## Step 1: Install Library

In [ ]:
!pip install requests beautifulsoup4 pandas lxml

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

print('Semua library siap!')

---
## Step 2: Pahami Struktur HTML

Sebelum scraping, kamu harus paham HTML.

HTML itu kerangka sebuah website. Setiap elemen punya tag:

```html
<html>
  <head>
    <title>Judul Halaman</title>
  </head>
  <body>
    <h1>Heading Besar</h1>
    <p class="review">Ini review filmnya...</p>
    <div id="content">
      <span class="rating">8/10</span>
    </div>
  </body>
</html>
```

Istilah penting:
- **Tag** -> nama elemen: `<p>`, `<div>`, `<span>`, `<h1>`
- **Class** -> penanda kelompok: `class="review"`
- **ID** -> penanda unik: `id="content"`
- **Attribute** -> info tambahan: `href`, `src`

**Cara lihat HTML website:**
Klik kanan di halaman web -> **Inspect** atau **Inspect Element**

---
## Step 3: Request - Ambil HTML dari Website

Library `requests` dipakai untuk mengunjungi website dan ambil HTML-nya.

Analogi: kayak buka browser dan ketik URL, tapi Python yang lakuin.

In [ ]:
# Coba request halaman IMDB
# Kita pakai review film The Shawshank Redemption

url = 'https://www.imdb.com/title/tt0111161/reviews/'

# Headers PENTING! Biar website kira kita browser, bukan bot
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9'
}

# Kirim request
response = requests.get(url, headers=headers)

print(f'Status Code: {response.status_code}')
print(f'  200 = Berhasil')
print(f'  403 = Diblokir')
print(f'  404 = Halaman tidak ditemukan')
print(f'Panjang HTML: {len(response.text)} karakter')
print()
print('Preview HTML (500 karakter pertama):')
print(response.text[:500])

---
## Step 4: Parse - Baca HTML dengan BeautifulSoup

HTML yang didapat berupa teks panjang yang berantakan.
`BeautifulSoup` ubah jadi struktur yang gampang dicari.

Analogi: kayak dapet buku tanpa daftar isi, terus ada yang bikinin daftar isi.

In [ ]:
# Parse HTML
soup = BeautifulSoup(response.text, 'lxml')

# Ambil judul halaman
judul = soup.find('title')
print(f'Judul halaman: {judul.text}')

# Cari semua elemen <h1>
semua_h1 = soup.find_all('h1')
print(f'Jumlah <h1>: {len(semua_h1)}')
for h1 in semua_h1:
    print(f'  - {h1.text.strip()}')

---
## Step 5: Explore - Cari Struktur Review

Sebelum extract data, kita cek dulu struktur HTML-nya.
IMDB sering update tampilan, jadi kita perlu adaptasi.

In [ ]:
# Cari class yang berhubungan dengan review
all_elements = soup.find_all(class_=True)
unique_classes = set()
for elem in all_elements:
    for cls in elem.get('class', []):
        unique_classes.add(cls)

review_classes = [c for c in unique_classes if 'review' in c.lower()]
print('Class yang berhubungan dengan review:')
for c in sorted(review_classes):
    count = len(soup.find_all(class_=c))
    print(f'  .{c} ({count} elemen)')

---
## Step 6: Extract - Ambil Data Review

Ini bagian inti! Kita ambil judul review, teks, rating, author, dan tanggal.

In [ ]:
def scrape_imdb_reviews(movie_id, max_reviews=50):
    '''Scraping review dari film IMDB.'''
    base_url = f'https://www.imdb.com/title/{movie_id}/reviews/'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept-Language': 'en-US,en;q=0.9'
    }

    print(f'Mengambil review: {base_url}')
    response = requests.get(base_url, headers=headers)

    if response.status_code != 200:
        print(f'Gagal! Status: {response.status_code}')
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, 'lxml')
    reviews = []

    # Coba beberapa class yang mungkin
    review_elements = soup.find_all('div', class_='imdb-user-review')
    if not review_elements:
        review_elements = soup.find_all('div', class_='review-container')
    if not review_elements:
        review_elements = soup.find_all('li', class_='ipl-content-list__item')

    print(f'Ditemukan {len(review_elements)} elemen review')

    for elem in review_elements[:max_reviews]:
        try:
            # Judul review
            judul_el = elem.find('a', class_='title') or elem.find(class_='title')
            judul = judul_el.text.strip() if judul_el else ''

            # Teks review
            teks_el = elem.find(class_='text') or elem.find('div', class_='content')
            teks = teks_el.text.strip() if teks_el else ''

            # Rating
            rating_el = elem.find(class_='rating-other-user-rating')
            rating = rating_el.text.strip() if rating_el else ''

            # Author
            author_el = elem.find(class_='display-name-link')
            author = author_el.text.strip() if author_el else ''

            # Tanggal
            date_el = elem.find(class_='review-date')
            date = date_el.text.strip() if date_el else ''

            if teks:
                reviews.append({
                    'judul_review': judul,
                    'teks_review': teks,
                    'rating': rating,
                    'author': author,
                    'tanggal': date,
                    'movie_id': movie_id
                })
        except Exception:
            continue

    df = pd.DataFrame(reviews)
    print(f'Berhasil mengambil {len(df)} review')
    return df

# Coba scraping!
df_reviews = scrape_imdb_reviews('tt0111161', max_reviews=25)
df_reviews.head()

---
## Step 7: Scraping Banyak Film

**PENTING: Kasih delay antar request biar tidak diblokir!**

In [ ]:
# Daftar film populer
film_list = [
    {'id': 'tt0111161', 'judul': 'The Shawshank Redemption'},
    {'id': 'tt0068646', 'judul': 'The Godfather'},
    {'id': 'tt0468569', 'judul': 'The Dark Knight'},
    {'id': 'tt0108052', 'judul': "Schindler's List"},
    {'id': 'tt0120737', 'judul': 'Lord of the Rings: Fellowship'},
]

semua_reviews = []

for i, film in enumerate(film_list):
    print(f'[{i+1}/{len(film_list)}] {film["judul"]} ({film["id"]})')
    df = scrape_imdb_reviews(film['id'], max_reviews=10)
    df['film'] = film['judul']
    semua_reviews.append(df)

    # DELAY 3 detik - jangan dihapus!
    if i < len(film_list) - 1:
        time.sleep(3)

df_semua = pd.concat(semua_reviews, ignore_index=True)
print(f'Total review: {len(df_semua)}')
df_semua.head()

---
## Step 8: Auto-Label Sentimen dari Rating

IMDB punya rating 1-10. Kita convert ke sentimen:
- Rating **7-10** -> Positif
- Rating **5-6** -> Netral
- Rating **1-4** -> Negatif

In [ ]:
def rating_ke_sentimen(rating_str):
    '''Convert rating IMDB ke label sentimen.'''
    try:
        angka = int(rating_str.split('/')[0].strip())
        if angka >= 7:
            return 'positif'
        elif angka >= 5:
            return 'netral'
        else:
            return 'negatif'
    except:
        return 'netral'

df_semua['sentimen'] = df_semua['rating'].apply(rating_ke_sentimen)

print('Distribusi sentimen:')
print(df_semua['sentimen'].value_counts())
print()

for s in ['positif', 'negatif', 'netral']:
    contoh = df_semua[df_semua['sentimen'] == s]
    if len(contoh) > 0:
        print(f'Contoh {s.upper()}:')
        print(f'  Rating: {contoh.iloc[0]["rating"]}')
        print(f'  Review: {contoh.iloc[0]["teks_review"][:150]}...')
        print()

---
## Step 9: Bersihkan Teks Review

In [ ]:
def bersihkan_review(teks):
    '''Bersihkan teks review.'''
    teks = re.sub(r'Read more.*$', '', teks)
    teks = re.sub(r'Warning: Spoilers.*$', '', teks)
    teks = re.sub(r'http\S+|www\S+', '', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    teks = re.sub(r'[^\w\s.,!?\'-]', '', teks)
    return teks

df_semua['review_bersih'] = df_semua['teks_review'].apply(bersihkan_review)
df_semua[['teks_review', 'review_bersih']].head()

---
## Step 10: Simpan ke CSV

In [ ]:
df_final = df_semua[['film', 'judul_review', 'review_bersih', 'rating', 'sentimen', 'author', 'tanggal']].copy()
df_final.columns = ['film', 'judul', 'review', 'rating', 'sentimen', 'author', 'tanggal']

df_final.to_csv('imdb_reviews_dataset.csv', index=False)
print(f'Dataset tersimpan: imdb_reviews_dataset.csv')
print(f'Total: {len(df_final)} review')
print()
print('Distribusi sentimen:')
print(df_final['sentimen'].value_counts())
print()
print('Per film:')
print(df_final['film'].value_counts())
df_final.head(10)

---
## Step 11: Visualisasi

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribusi sentimen
counts = df_final['sentimen'].value_counts()
colors = ['#2ecc71', '#95a5a6', '#e74c3c']
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=colors)
axes[0].set_title('Distribusi Sentimen')

# Review per film
film_counts = df_final['film'].value_counts()
axes[1].barh(film_counts.index, film_counts.values, color='steelblue')
axes[1].set_title('Review per Film')

# Panjang review
df_final['panjang'] = df_final['review'].str.len()
sns.boxplot(data=df_final, x='sentimen', y='panjang', ax=axes[2], palette=colors)
axes[2].set_title('Panjang Review per Sentimen')

plt.tight_layout()
plt.show()

---
## Bonus: Selenium (Untuk Website Dinamis)

Kalau `requests` + `BeautifulSoup` tidak bisa ambil data (website pakai JavaScript), pakai **Selenium**.

Selenium buka browser asli dan kontrol secara otomatis.

```python
!pip install selenium

from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get('https://www.imdb.com/title/tt0111161/reviews/')
time.sleep(3)

# Scroll untuk load lebih banyak review
driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
time.sleep(2)

# Ambil HTML setelah JavaScript render
html = driver.page_source
soup = BeautifulSoup(html, 'lxml')

driver.quit()
```

Selenium lebih lambat tapi bisa handle website yang `requests` tidak bisa.

---
## Etika Web Scraping

### 1. Cek robots.txt
Setiap website punya `robots.txt`: https://www.imdb.com/robots.txt

### 2. Jangan spam request
Kasih delay minimal 2-3 detik antar request.

### 3. Pakai User-Agent yang benar
Identifikasi diri sebagai developer, bukan bot jahat.

### 4. Hormati Terms of Service
Cek apakah scraping diperbolehkan untuk penggunaan kamu.

### 5. Cache hasil scraping
Simpan hasil, jangan scrape halaman yang sama berulang kali.

---
## Ringkasan

```
1. Tentukan target (website apa, data apa)
2. Inspect HTML (klik kanan -> Inspect)
3. Request (requests.get) -> dapat HTML
4. Parse (BeautifulSoup) -> struktur yang bisa dicari
5. Extract (find/find_all) -> ambil data
6. Bersihkan (regex) -> hapus karakter tidak perlu
7. Simpan (pandas -> CSV) -> dataset siap pakai
```

## Cheat Sheet BeautifulSoup

```python
# Cari satu elemen
soup.find('tag_name')
soup.find(class_='nama_class')
soup.find(id='nama_id')

# Cari semua elemen
soup.find_all('p')
soup.find_all('div', class_='review')

# Ambil teks
element.text.strip()

# Ambil attribute
element['href']
element.get('src')
```

## Langkah Selanjutnya

File `imdb_reviews_dataset.csv` bisa dipakai di:
- `sentiment_analysis_tutorial.ipynb` -> ganti dataset
- `03_csv_dataset.ipynb` -> load CSV hasil scraping